# Лабораторна робота №3
## Навчання агента з підкріпленням для керування посадкою космічного апарата

**Дисципліна:** Штучний інтелект в ігрових застосунках  
**Середовище:** Gymnasium LunarLander-v3  
**Алгоритм:** DQN (Deep Q-Network) через Stable-Baselines3


## Крок 1: Встановлення залежностей

Встановимо необхідні бібліотеки. SWIG потрібен для компіляції Box2D — фізичного рушія середовища.

In [ ]:
!pip install --quiet swig
!pip install --quiet "gymnasium[box2d]" stable-baselines3 matplotlib numpy
print('Встановлення завершено.')

## Крок 2: Дослідження середовища LunarLander

Створимо середовище та дослідимо його:
- **Простір спостережень**: ??-вимірний вектор (x, y, швидкості, кут, контакт ніг)
- **Простір дій**: ?? дискретні дії (нічого, лівий двигун, головний двигун, правий двигун)
- **Винагорода**: +100..+140 за посадку, -100 за аварію, штрафи за використання двигунів

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Створення середовища
env = gym.make("LunarLander-v3", render_mode="rgb_array")

print(f"Простір спостережень: {env.observation_space}")
print(f"Простір дій: {env.action_space}")
print(f"\nОпис спостережень:")
print(f"  [0] X координата")
print(f"  [1] Y координата")
print(f"  [2] Горизонтальна швидкість")
print(f"  [3] Вертикальна швидкість")
print(f"  [4] Кут нахилу")
print(f"  [5] Кутова швидкість")
print(f"  [6] Контакт лівої ноги (0/1)")
print(f"  [7] Контакт правої ноги (0/1)")
print(f"\nДії: 0=нічого, 1=лівий двигун, 2=головний двигун, 3=правий двигун")

### Випадковий агент

Запустимо декілька епізодів з випадковим агентом, щоб побачити, як виглядає "ненавчена" поведінка.

In [ ]:
# Оцінка випадкового агента (10 епізодів)
random_rewards = []
for episode in range(10):
    obs, info = env.reset(seed=episode)
    total_reward = 0
    terminated, truncated = False, False
    while not (terminated or truncated):
        action = env.action_space.sample()  # випадкова дія
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
    random_rewards.append(total_reward)

print(f"Випадковий агент (10 епізодів):")
print(f"  Середня винагорода: {np.mean(random_rewards):.1f}")
print(f"  Мін / Макс: {np.min(random_rewards):.1f} / {np.max(random_rewards):.1f}")
print(f"\nЯк бачимо, випадковий агент отримує негативну винагороду (зазвичай від -100 до -300).")

### Візуалізація одного епізоду випадкового агента

In [ ]:
# Запис одного епізоду для візуалізації
frames = []
obs, info = env.reset(seed=42)
terminated, truncated = False, False
while not (terminated or truncated):
    frames.append(env.render())
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
frames.append(env.render())

# Анімація
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
img = ax.imshow(frames[0])

def animate(i):
    img.set_data(frames[i])
    return [img]

anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
plt.close()
HTML(anim.to_html5_video())

## Крок 3: Теоретична основа

### Q-навчання та DQN

**Q-функція** $Q(s, a)$ оцінює якість пари "стан-дія" — очікувану кумулятивну винагороду.

**Рівняння Беллмана:**
$$Q^*(s, a) = R(s, a) + \gamma \cdot \max_{a'} Q^*(s', a')$$

**DQN** використовує нейронну мережу для апроксимації Q-функції, коли простір станів занадто великий для таблиці.

Ключові компоненти DQN:
- **Experience Replay**: зберігає переходи $(s, a, r, s')$ у буфері, навчається на випадкових міні-батчах
- **Target Network**: окрема копія мережі для стабільного навчання
- **Epsilon-greedy**: з ймовірністю $\varepsilon$ обирає випадкову дію (дослідження), інакше — найкращу (використання)

### Stable-Baselines3

SB3 надає готову реалізацію DQN. Основні гіперпараметри:
- `learning_rate` — швидкість навчання нейронної мережі
- `buffer_size` — розмір буфера досвіду
- `learning_starts` — кількість кроків до початку навчання
- `batch_size` — розмір міні-батчу для навчання
- `exploration_fraction` — частка навчання з epsilon-greedy дослідженням
- `exploration_final_eps` — фінальне значення epsilon

## TODO 1: Обчислення індивідуальних параметрів

Порахуйте ваш параметр **A** та визначте фізичні параметри вашого індивідуального середовища.

Український алфавіт: А=1, Б=2, В=3, Г=4, Ґ=5, Д=6, Е=7, Є=8, Ж=9, З=10, И=11, І=12, Ї=13, Й=14, К=15, Л=16, М=17, Н=18, О=19, П=20, Р=21, С=22, Т=23, У=24, Ф=25, Х=26, Ц=27, Ч=28, Ш=29, Щ=30, Ь=31, Ю=32, Я=33

In [ ]:
# TODO: Замініть значення A на ваш розрахований параметр
# A = (номер першої літери імені) + (номер першої літери прізвища)
# Приклад: Олександр Бауск -> О=19, Б=2 -> A = 21

A = '???'  # TODO: ваше значення A

# Індивідуальні параметри середовища
individual_gravity = -10.0 + (A % 5) * (-0.5)
individual_wind = A > 15
individual_wind_power = (A % 10) * 1.5
individual_turbulence = (A % 7) * 0.25

print(f"Параметр A = {A}")
print(f"\nІндивідуальні параметри середовища:")
print(f"  gravity = {individual_gravity}")
print(f"  enable_wind = {individual_wind}")
print(f"  wind_power = {individual_wind_power}")
print(f"  turbulence_power = {individual_turbulence}")

# Оцінка складності
difficulty = abs(individual_gravity) + (individual_wind_power if individual_wind else 0) + individual_turbulence * 5
print(f"\nОцінка складності: {difficulty:.1f} (чим більше, тим складніше)")

## TODO 2: Навчання базового DQN-агента

Натренуємо DQN-агента на **стандартному** середовищі (gravity=-10, без вітру).
Це дозволить частково порівняти результати всіх студентів на рівних умовах.

Для відстеження прогресу навчання використаємо callback, який записує винагороду кожного епізоду.

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy

class RewardLoggerCallback(BaseCallback):
    """Callback для запису винагород під час навчання."""
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.current_rewards = 0

    def _on_step(self):
        self.current_rewards += self.locals['rewards'][0]
        if self.locals['dones'][0]:
            self.episode_rewards.append(self.current_rewards)
            self.current_rewards = 0
        return True

In [ ]:
# Стандартне середовище (однакове для всіх студентів)
env_standard = gym.make("LunarLander-v3")

# TODO: Спробуйте різні значення total_timesteps (150000, 200000, 250000)
# Більше кроків = краще навчання, але довше тренування
TOTAL_TIMESTEPS = 'TODO' # TODO: виберіть кількість кроків

default_hyperparams = {
    "learning_rate": 1e-4,
    "buffer_size": 1000000,
    "exploration_fraction": 0.1,
    "exploration_initial_eps": 1.0,
    "exploration_final_eps": 0.05,
    "batch_size": 32,
    "target_update_interval": 10000
}

# Створення DQN-агента з параметрами за замовчуванням
model_standard = DQN(
    "MlpPolicy",
    env_standard,
    verbose=0,
    seed=A,  # Використовуємо A як seed для відтворюваності
    **default_hyperparams
)

# Callback для запису прогресу
callback = RewardLoggerCallback()

# Навчання!
print(f"Початок навчання ({TOTAL_TIMESTEPS} кроків)...")
model_standard.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback)
print(f"Навчання завершено! Епізодів: {len(callback.episode_rewards)}")

В коді вище default_hyperparams -- це гіперпараметри тренування моделі за замовчуванням (дефолтні), які вам треба підібрати для кращого тренування вашої стандартної моделі, а трохи пізніше -- вашої індивідуальної моделі (яка відрізняється від стандартної моделі параметрами середовища).

Почитайте документацію про гіперпараметри, щоб мати уявлення про їх різномаїтність для навчання з підкріпленням і важливість впливу гіперпараметрів на якість тренування моделі.

https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#stable_baselines3.dqn.DQN

### Графік навчання

Побудуємо графік винагороди по епізодах. Згладжена крива (rolling average) показує загальну тенденцію навчання.

In [ ]:
def plot_training_curve(episode_rewards, title="Прогрес навчання DQN", window=50):
    """Графік винагороди по епізодах зі згладжуванням."""
    rewards = np.array(episode_rewards)
    fig, ax = plt.subplots(figsize=(12, 5))

    # Сирі винагороди (напівпрозорі)
    ax.plot(rewards, alpha=0.3, color='blue', linewidth=0.5, label='Винагорода за епізод')

    # Згладжена крива (rolling average)
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color='red', linewidth=2,
                label=f'Середнє за {window} епізодів')

    ax.axhline(y=200, color='green', linestyle='--', alpha=0.7, label='Поріг "розв\'язано" (200)')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Епізод')
    ax.set_ylabel('Сумарна винагорода')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_training_curve(callback.episode_rewards, "Базовий DQN — стандартне середовище")

## TODO 3: Оцінка базового агента

Оцінимо натренованого агента на 100 епізодах і візуалізуємо траєкторію посадки.

In [ ]:
# Оцінка на 100 епізодах
eval_env = gym.make("LunarLander-v3")
mean_reward, std_reward = evaluate_policy(model_standard, eval_env, n_eval_episodes=100)
print(f"Оцінка базового агента (100 епізодів):")
print(f"  Середня винагорода: {mean_reward:.1f} +/- {std_reward:.1f}")
print(f"  Статус: {'РОЗВ\'ЯЗАНО!' if mean_reward >= 200 else 'Ще не розв\'язано (потрібно >= 200)'}")
eval_env.close()

### Візуалізація траєкторії посадки

Записуємо позицію апарата (X, Y) протягом епізоду та будуємо графік траєкторії.

In [ ]:
def record_trajectory(model, env, seed=42):
    """Запис траєкторії (x, y) під час одного епізоду."""
    obs, info = env.reset(seed=seed)
    trajectory = []
    total_reward = 0
    terminated, truncated = False, False

    while not (terminated or truncated):
        x, y = obs[0], obs[1]
        trajectory.append([x, y])
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

    trajectory.append([obs[0], obs[1]])
    return np.array(trajectory), total_reward


def plot_trajectory(trajectory, total_reward, title="Траєкторія посадки"):
    """Візуалізація траєкторії посадки."""
    fig, ax = plt.subplots(figsize=(8, 6))

    # Траєкторія
    ax.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=2, alpha=0.7, label='Траєкторія')
    ax.scatter(trajectory[0, 0], trajectory[0, 1], color='green', s=150, zorder=5,
               marker='^', label='Старт')
    ax.scatter(trajectory[-1, 0], trajectory[-1, 1], color='red', s=150, zorder=5,
               marker='v', label='Посадка')

    # Посадковий майданчик
    ax.axhline(y=0, color='brown', linestyle='-', alpha=0.5)
    ax.axvspan(-0.2, 0.2, ymin=0, ymax=0.05, alpha=0.3, color='yellow', label='Посадковий майданчик')

    ax.set_xlabel('X координата')
    ax.set_ylabel('Y координата')
    ax.set_title(f"{title}\nВинагорода: {total_reward:.1f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Запис та візуалізація
traj_env = gym.make("LunarLander-v3")
traj, rew = record_trajectory(model_standard, traj_env)
plot_trajectory(traj, rew, "Базовий DQN — стандартне середовище")
traj_env.close()

### Анімація посадки натренованого агента

In [ ]:
# Анімація одного епізоду натренованого агента
anim_env = gym.make("LunarLander-v3", render_mode="rgb_array")
obs, info = anim_env.reset(seed=42)
frames_trained = []
terminated, truncated = False, False

while not (terminated or truncated):
    frames_trained.append(anim_env.render())
    action, _ = model_standard.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = anim_env.step(action)
frames_trained.append(anim_env.render())
anim_env.close()

fig, ax = plt.subplots(figsize=(6, 4))
ax.axis('off')
img = ax.imshow(frames_trained[0])

def animate(i):
    img.set_data(frames_trained[i])
    return [img]

anim = animation.FuncAnimation(fig, animate, frames=len(frames_trained), interval=50, blit=True)
plt.close()
HTML(anim.to_html5_video())

## TODO 4: Експерименти з гіперпараметрами

Проведіть **експерименти** з різними гіперпараметрами DQN.

Приклади гіперпараметрів для експериментів:
- `learning_rate`: 1e-4, 5e-4, 1e-3
- `buffer_size`: 10000, 50000, 100000
- `exploration_fraction`: 0.1, 0.2, 0.5

Для кожного експерименту запишіть винагороду та побудуйте графік.

In [ ]:
# TODO: Визначте 3+ конфігурації гіперпараметрів
# Замініть '''TODO''' на конкретні значення

experiments = {
    "baseline": {
        **default_hyperparams,
        "learning_rate": 1e-4,     # значення за замовчуванням
        "buffer_size": 50000,
        "exploration_fraction": 0.1,
    },
    "experiment_1": {
        **default_hyperparams,
        "learning_rate": '''TODO''',       # TODO: спробуйте інше значення
        "buffer_size": '''TODO''',          # TODO: спробуйте інше значення
        "exploration_fraction": '''TODO''',  # TODO: спробуйте інше значення
    },
    "experiment_2": {
        **default_hyperparams,
        "learning_rate": '''TODO''',       # TODO: спробуйте інше значення
        "buffer_size": '''TODO''',          # TODO: спробуйте інше значення
        "exploration_fraction": '''TODO''',  # TODO: спробуйте інше значення
        "exploration_final_eps": '''TODO''', # TODO: спробуйте інше значення
        "batch_size": '''TODO''',            # TODO: спробуйте інше значення
        "target_update_interval": '''TODO''' # TODO: спробуйте інше значення
    },
    # Можете за необхідності додати experiment_3, experiment_4
}

print("Конфігурації експериментів:")
for name, params in experiments.items():
    print(f"  {name}: {params}")

In [ ]:
# Навчання та оцінка для кожної конфігурації
# (Увага: це може зайняти 15-30 хвилин для 3 експериментів)

EXPERIMENT_TIMESTEPS = 200000  # Менше кроків для швидшого порівняння, більше -- для довшого і, можливо, кращої якості

def run_experiments(experiments_spec, environment_params = None, timesteps = EXPERIMENT_TIMESTEPS):
    trained_models = {}
    results = {}
    for name, params in experiments_spec.items():
        print(f"\n{'='*50}")
        print(f"Навчання: {name}")
        print(f"Параметри: {params}")

        if environment_params is None:
            exp_env = gym.make("LunarLander-v3")
        else:
            exp_env = gym.make(
                "LunarLander-v3",
                **environment_params
            )
        model_exp = DQN(
            "MlpPolicy",
            exp_env,
            verbose=0,
            seed=A,
            **params
        )

        cb = RewardLoggerCallback()
        model_exp.learn(total_timesteps=timesteps, callback=cb)

        # Оцінка
        if environment_params is None:
            eval_env = gym.make("LunarLander-v3")
        else:
            eval_env = gym.make(
                "LunarLander-v3",
                **environment_params
            )
        mean_r, std_r = evaluate_policy(model_exp, eval_env, n_eval_episodes=50)
        eval_env.close()
        exp_env.close()

        results[name] = {
            "rewards": cb.episode_rewards,
            "mean_reward": mean_r,
            "std_reward": std_r,
            "params": params,
        }
        trained_models[name] = model_exp

        print(f"Результат: {mean_r:.1f} +/- {std_r:.1f}")

    print(f"\n{'='*50}")
    print("Всі експерименти завершено!")
    return results, trained_models

experiment_results, trained_models = run_experiments(
    experiments
)

Змінна trained_models має ваші моделі -- на етапі порівняння з індивідуальною моделлю ви її використаєте

### Порівняння експериментів

In [ ]:
# Порівняльний графік навчання
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Графік 1: Криві навчання
colors = ['blue', 'orange', 'green', 'red', 'purple']
for idx, (name, result) in enumerate(experiment_results.items()):
    rewards = np.array(result["rewards"])
    window = 30
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0].plot(smoothed, color=colors[idx % len(colors)], linewidth=2, label=name)

axes[0].axhline(y=200, color='green', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Епізод')
axes[0].set_ylabel('Винагорода (згладжена)')
axes[0].set_title('Порівняння кривих навчання')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Графік 2: Стовпчикова діаграма фінальних результатів
names = list(experiment_results.keys())
means = [experiment_results[n]["mean_reward"] for n in names]
stds = [experiment_results[n]["std_reward"] for n in names]
bar_colors = [colors[i % len(colors)] for i in range(len(names))]

axes[1].bar(names, means, yerr=stds, color=bar_colors, alpha=0.7, capsize=5)
axes[1].axhline(y=200, color='green', linestyle='--', alpha=0.5, label='Поріг 200')
axes[1].set_ylabel('Середня винагорода (50 епізодів)')
axes[1].set_title('Порівняння результатів')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Таблиця результатів
print(f"\n{'Конфігурація':<20} {'Сер. винагорода':>15} {'Std':>10}")
print('-' * 50)
for name, result in experiment_results.items():
    print(f"{name:<20} {result['mean_reward']:>15.1f} {result['std_reward']:>10.1f}")

На цьому етапі вам треба поекспериментувати з гіперпараметрами, щоб знайти краще рішення, ніж **дефолтні** значення.

Спробуйте різні набори гіперпараметрів в ваших експериментах, а також іншу (зазвичай допомагає взяти більшу) кількість ітерацій в EXPERIMENT_TIMESTEPS.

Найкращий отриманий вами результат запишіть в оптимальний набір гіперпараметрів, який ви завантажите в турнірну таблицю:

In [ ]:
optimal_hyperparams = {
    # TODO: скопіюємо в цей обʼєкт значення з найкращого експерименту:
    **experiments['TODO']  # Наприклад 'experiment_1', якщо найкращим був experiment_1
}

from json import dumps
print(dumps(optimal_hyperparams)) # Строка для завантаження в турнірну таблицю

Результат виконання ячейки вище знадобиться вам для завантаження в турнірну таблицю.

## TODO 5: Навчання на індивідуальному середовищі

Тепер натренуємо агента на вашому **індивідуальному** середовищі з унікальними фізичними параметрами.

Зверніть увагу: якщо у вас увімкнено вітер або сильну гравітацію, навчання може потребувати більше кроків або іншого підбору гіперпараметрів.

**УВАГА:** ОПТИМАЛЬНІ гіперпараметри можуть бути іншими для вашого індивідуального середовища.

Ваші гіперпараметри, отримані на кроку вище, можуть спрацювати достатньо добре для вашого середовища.

Якщо ви хочете отримати найкращий результат, спробуйте знову поекспериментувати з гіперпараметрами вже для вашого середовища env_individual,
для чого треба або спробувати різні individual_hyperparams, або (це важчий варіант) створити обʼєкт з кількома різними наборами гіперпараметрів (як вище для стандартного середовища) і передати його в функцію run_experiments разом з вашими індивідуальними параметрами середовища.

In [ ]:
individual_hyperparams = {
    **optimal_hyperparams,  # Тут ми просто копіюємо результат найкращого експерименту з "стандартного" середовища
    # За бажанням підберіть інші гіперпараметри для вашого індивідуального середовища
}
print(individual_hyperparams)

In [ ]:
#**Ускладнений** варіант для студентів з досвідом роботи з Python, це робити необовʼязково:
#Можна використати функцію пошуку оптимальних гіперпараметрів
#individual_experiments = ???
#results = run_experiments(
#    individual_experiments,
#    environment_params = ???, # Ваші індивідуальні параметри середовища
#    timesteps = INDIVIDUAL_TIMESTEPS
#)

# Створення індивідуального середовища з вашими параметрами гравітації і вітру:
env_individual = gym.make(
    "LunarLander-v3",
    gravity=individual_gravity,
    enable_wind=individual_wind,
    wind_power=individual_wind_power,
    turbulence_power=individual_turbulence,
)

print(f"Індивідуальне середовище:")
print(f"  gravity = {individual_gravity}")
print(f"  enable_wind = {individual_wind}")
print(f"  wind_power = {individual_wind_power}")
print(f"  turbulence_power = {individual_turbulence}")

# TODO: Виберіть гіперпараметри для індивідуального середовища
# Підказка: якщо у вас складне середовище (сильний вітер/гравітація),
# спробуйте збільшити кількість кроків або exploration_fraction

INDIVIDUAL_TIMESTEPS = 200000  # TODO: кількість кроків (наприклад, 200000 або 300000)

model_individual = DQN(
    "MlpPolicy",
    env_individual,
    verbose=0,
    seed=A,
    **individual_hyperparams
)

callback_ind = RewardLoggerCallback()
print(f"\nПочаток навчання ({INDIVIDUAL_TIMESTEPS} кроків)...")

# Використайте реіниціалізацію pytorch у випадку помилок
# import torch
# torch.set_grad_enabled(True)
model_individual.learn(total_timesteps=INDIVIDUAL_TIMESTEPS, callback=callback_ind)
print(f"Навчання завершено! Епізодів: {len(callback_ind.episode_rewards)}")

In [ ]:
# Графік навчання на індивідуальному середовищі
plot_training_curve(callback_ind.episode_rewards, "DQN — індивідуальне середовище")

# Оцінка
eval_ind_env = gym.make(
    "LunarLander-v3",
    gravity=individual_gravity,
    enable_wind=individual_wind,
    wind_power=individual_wind_power,
    turbulence_power=individual_turbulence,
)
mean_r_ind, std_r_ind = evaluate_policy(model_individual, eval_ind_env, n_eval_episodes=100)
eval_ind_env.close()

print(f"\nОцінка на індивідуальному середовищі (100 епізодів):")
print(f"  Середня винагорода: {mean_r_ind:.1f} +/- {std_r_ind:.1f}")

## TODO 6: Фінальна візуалізація

Створіть порівняльну візуалізацію: траєкторії посадки на стандартному та індивідуальному середовищах.

In [ ]:
# Порівняння траєкторій
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Стандартне середовище
std_env = gym.make("LunarLander-v3")
# Тут ми беремо найкращу стандартну модель з ваших експериментів
model_standard_optimized = trained_models['TODO'] # Наприклад 'experiment_1', якщо найкращим був experiment_1

traj_std, rew_std = record_trajectory(model_standard_optimized, std_env)
std_env.close()

axes[0].plot(traj_std[:, 0], traj_std[:, 1], 'b-', linewidth=2)
axes[0].scatter(traj_std[0, 0], traj_std[0, 1], color='green', s=100, marker='^', zorder=5)
axes[0].scatter(traj_std[-1, 0], traj_std[-1, 1], color='red', s=100, marker='v', zorder=5)
axes[0].axhline(y=0, color='brown', linestyle='-', alpha=0.3)
axes[0].set_title(f"Стандартне середовище\nВинагорода: {rew_std:.1f}")
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)

# Індивідуальне середовище
ind_env = gym.make(
    "LunarLander-v3",
    gravity=individual_gravity,
    enable_wind=individual_wind,
    wind_power=individual_wind_power,
    turbulence_power=individual_turbulence,
)
traj_ind, rew_ind = record_trajectory(model_individual, ind_env)
ind_env.close()

axes[1].plot(traj_ind[:, 0], traj_ind[:, 1], 'r-', linewidth=2)
axes[1].scatter(traj_ind[0, 0], traj_ind[0, 1], color='green', s=100, marker='^', zorder=5)
axes[1].scatter(traj_ind[-1, 0], traj_ind[-1, 1], color='red', s=100, marker='v', zorder=5)
axes[1].axhline(y=0, color='brown', linestyle='-', alpha=0.3)
wind_str = f", вітер={individual_wind_power:.1f}" if individual_wind else ""
axes[1].set_title(f"Індивідуальне (g={individual_gravity}{wind_str})\nВинагорода: {rew_ind:.1f}")
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"Порівняння траєкторій посадки (A={A})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## TODO 7: Збереження моделей для турнірної таблиці

Збережіть обидві натреновані моделі. Ці файли потрібно буде здати разом зі звітом для участі в турнірній таблиці.

In [ ]:
# Збереження моделей
model_standard_optimized.save("model_standard")
model_individual.save("model_individual")

print("Моделі збережено:")
print("  model_standard.zip — для турнірної таблиці (стандартне середовище)")
print("  model_individual.zip — для індивідуального рейтингу")

# Перевірка: завантажити та оцінити збережену модель
loaded_model = DQN.load("model_standard")
check_env = gym.make("LunarLander-v3")
check_mean, check_std = evaluate_policy(loaded_model, check_env, n_eval_episodes=10)
check_env.close()
print(f"\nПеревірка завантаженої моделі: {check_mean:.1f} +/- {check_std:.1f}")

## Підсумкова таблиця результатів

Заповніть таблицю на основі ваших результатів:

In [ ]:
# Фінальний звіт
print("=" * 60)
print(f"ПІДСУМОК ЛАБОРАТОРНОЇ РОБОТИ №3")
print(f"Параметр A = {A}")
print("=" * 60)
print(f"\nСтандартне середовище:")
print(f"  Середня винагорода: {mean_reward:.1f} +/- {std_reward:.1f}")
print(f"  Статус: {'РОЗВ\'ЯЗАНО' if mean_reward >= 200 else 'Не розв\'язано'}")
print(f"\nІндивідуальне середовище (g={individual_gravity}, wind={individual_wind}):")
print(f"  Середня винагорода: {mean_r_ind:.1f} +/- {std_r_ind:.1f}")
print(f"\nЕксперименти з гіперпараметрами:")
for name, result in experiment_results.items():
    print(f"  {name}: {result['mean_reward']:.1f} +/- {result['std_reward']:.1f}")
print(f"\nНайкраща конфігурація: {max(experiment_results, key=lambda k: experiment_results[k]['mean_reward'])}")
print("=" * 60)